In [ ]:
from IPython import get_ipython
ipython = get_ipython()
if ipython is not None:
    ipython.run_line_magic('load_ext', 'autoreload')
    ipython.run_line_magic('autoreload', '2')
else:
    print("could not load extension")

%reload_ext autoreload

In [ ]:
import getpass

user = getpass.getuser()
from omni.isaac.kit import SimulationApp

simulation_app = SimulationApp({"livesync_usd": f"omniverse://localhost/Users/{user}/telecom_test.usd"})

In [ ]:
import carb
from omni.physx import get_physx_scene_query_interface  # for raycasting e.g raycast_closest()
import omni.physx.scripts.utils as physx_utils
from omni.isaac.core import World
from omni.isaac.core.objects import DynamicCuboid, DynamicSphere, DynamicCone, DynamicCylinder
from omni.timeline import get_timeline_interface
from omni.isaac.core.utils.stage import get_current_stage
import omni.isaac.core.utils.prims as prims_utils
import omni.kit.commands

import numpy as np
import matplotlib.pyplot as plt
from pprint import pprint

In [ ]:
from omni.syntheticdata import visualize
from omni.kit.viewport.utility import get_active_viewport
import omni.replicator.core as rep
from omni.isaac.core.utils.viewports import set_camera_view


set_camera_view(eye=np.array([0, 0, 1000]), target=np.array([0, 0, 0]))

viewport_api = get_active_viewport()
active_cam = viewport_api.get_active_camera()

viewport_api.set_texture_resolution([1000, 1000])
resolution = viewport_api.get_texture_resolution()
render_product = rep.create.render_product(active_cam, resolution)

In [ ]:
rgb = rep.AnnotatorRegistry.get_annotator("rgb")
rgb.attach([render_product])
depth = rep.AnnotatorRegistry.get_annotator("distance_to_image_plane")
depth.attach([render_product])
semantic_segmentation = rep.AnnotatorRegistry.get_annotator("semantic_segmentation")
semantic_segmentation.attach([render_product])

# Setup Test World Programatically


## Test case 1: Lunar environment + two antennas

In [ ]:
world = World(stage_units_in_meters=1.0)
world.clear()

In [ ]:
usd_path = f"omniverse://localhost/Users/ubuntu/moon_environnement.usd"
prim_path = "/World/Lunar_Base"

ground_path = prim_path + "/Ground"
# create the prim
prim_ground = prims_utils.create_prim(prim_path=ground_path, usd_path=usd_path)
world.render()

In [ ]:
from AntennaClass import Antenna
### Creation of the first antenna, with a gain = 10dB, a waveLength = 1e-3m, an efficiency = 1

antenna1 = world.scene.add(
    Antenna(
        prim_path="/World/antenna1",
        name="ant1",
        waveLength = 1e-3,
        G = 10,
        e = 1,
        position=np.array([12,-21,23]),
        scale=np.array([0.1,0.1,4]),
        height=1.0,
        color=np.array([0.0,1.0,0.0]),
        power = 100
    )
)
world.render()

In [ ]:
### Creation of the second antenna, with the same properties as the first one

antenna2 = world.scene.add(
    Antenna(
        prim_path="/World/antenna2",
        name="ant2",
        waveLength = 1e-3,
        G = 10,
        e = 1,
        position=np.array([-56,9,21]),
        scale=np.array([0.1,0.1,4]),
        height=1.0,
        color=np.array([1.0,0.0,0.0]),
        power = 1e-6
    )
)
world.render()

In [ ]:
# the Sun
stage = get_current_stage()
def adjust_light(stage, position):
    lights = []
    for prim in stage.Traverse():
        if prim.GetTypeName() == "SphereLight":
            lights.append(prim)
    if lights:
        light = lights[0]
        light_path = light.GetPath()
        # Adjust position
        translate_attr = stage.GetPrimAtPath(light_path).GetAttribute("xformOp:translate")
        translate_attr.Set(position)
        # Adjust intensity
        intensity_attr = stage.GetPrimAtPath(light_path).GetAttribute("intensity")
        intensity_attr.Set(3e6)

In [ ]:
sun_coord = (5, -5, 5)
adjust_light(stage, sun_coord)
world.render()


###############################

# Run the application for multiple frames to ensure the synthetic data pipeline is initialized
timeline = get_timeline_interface()
timeline.play()
for _ in range(10):
    simulation_app.update()
timeline.pause()

# Get groundtruth
rgb_data = rgb.get_data()
depth_data = depth.get_data()
semantic_segmentation_data = semantic_segmentation.get_data()

###############################

# Initialize grid with proper resolution
cell_size = 0.025  # meters
grid_size = int(10 / cell_size)   # e.g 100x100 grid for 0.1 cell size in a 10x10 area
half_side_size = grid_size/2*cell_size 
grid_size ** 2

In [ ]:
stage = get_current_stage()
distance_increment = 0.5

# Perform raycast from antenna1 to antenna2
output_ray = antenna1.performRaycast(antenna2, stage, distance_increment)
print(output_ray)

# Check if the raycast hit the correct target
antenna1.displayRaycastCoherence(output_ray, antenna2.prim_path)

# Check if the power is sufficient at the given distance
antenna1.displayPowerFeasibility(output_ray, antenna2)

## Test case 2: antennas + obstacle

In [ ]:
world = World(stage_units_in_meters=1.0)
world.clear()

In [ ]:
usd_path = f"omniverse://localhost/Users/ubuntu/moon_environnement.usd"
prim_path = "/World/Lunar_Base"

ground_path = prim_path + "/Ground"
# create the prim
prim_ground = prims_utils.create_prim(prim_path=ground_path, usd_path=usd_path)
world.render()

In [ ]:
from AntennaClass import Antenna

### Creation of the first antenna, with a gain = 10dB, a waveLength = 1e-3m, an efficiency = 1

antenna1 = world.scene.add(
    Antenna(
        prim_path="/World/antenna1",
        name="antenna1",
        waveLength = 1e-3,
        G = 10,
        e = 1,
        position=np.array([12,-21,23]),
        scale=np.array([0.1,0.1,4]),
        height=1.0,
        color=np.array([0.0,1.0,0.0]),
        power = 100
    )
)
world.render()

In [ ]:
### Creation of the second antenna, with the same properties as the first one

antenna2 = world.scene.add(
    Antenna(
        prim_path="/World/antenna2",
        name="antenna2",
        waveLength = 1e-3,
        G = 10,
        e = 1,
        position=np.array([-56,9,21]),
        scale=np.array([0.1,0.1,4]),
        height=1.0,
        color=np.array([1.0,0.0,0.0]),
        power = 1e-6
    )
)
world.render()

In [ ]:
### Creation of a cube between the two antennas, to act as an obstacle

obstacle = world.scene.add(
    DynamicCuboid(
        prim_path="/World/obstacle",
        name="obstacle",
        position=np.array([-20, -3, 26]),
        scale=np.array([1, 41, 15]),
        size=1.,
        color=np.array([0, 0, 1]),
    )
)
world.render()

In [ ]:
# Run the application for multiple frames to ensure the synthetic data pipeline is initialized
timeline = get_timeline_interface()
timeline.play()
for _ in range(10):
    simulation_app.update()
timeline.pause()

# Get groundtruth
rgb_data = rgb.get_data()
depth_data = depth.get_data()
semantic_segmentation_data = semantic_segmentation.get_data()

In [ ]:
stage = get_current_stage()
distance_increment = 0.5

# Perform raycast from antenna1 to antenna2
output_ray = antenna1.performRaycast(antenna2, stage, distance_increment)

# Check if the raycast hit the correct target
antenna1.displayRaycastCoherence(output_ray, antenna2.prim_path)

# Check if the power is sufficient at the given distance
antenna1.displayPowerFeasibility(output_ray, antenna2)

In [ ]:
'''
origin = antenna1
pointingObject = antenna2
pointingObjectPath = antenna2.prim_path
origin = moveOriginRaycast(antenna1, antenna2, 0.3)

stage = get_current_stage()
out_raycast = performRaycast(stage, antenna1, antenna2, origin)
print(out_raycast)

displayRaycastCoherence(out_raycast, pointingObjectPath)
displayPowerFeasibility(out_raycast[1], antenna2.getPower)'''

## Mapping of the line of sight

In [ ]:
world = World(stage_units_in_meters=1.0)
world.clear()

In [ ]:
usd_path = f"omniverse://localhost/Users/ubuntu/moon_environnement.usd"
prim_path = "/World/Lunar_Base"

ground_path = prim_path + "/Ground"
# create the prim
prim_ground = prims_utils.create_prim(prim_path=ground_path, usd_path=usd_path)

### Creation of the first antenna, with a gain = 10dB, a waveLength = 1e-3m, an efficiency = 1

emitter = world.scene.add(
    Antenna(
        prim_path="/World/emitter",
        name="emitter",
        waveLength = 1e-3,
        G = 10,
        e = 1,
        position=np.array([0,0,12]),
        scale=np.array([0.1,0.1,4]),
        height=1.0,
        color=np.array([0.0,1.0,0.0]),
    )
)

receiver = world.scene.add(
    Antenna(
        prim_path="/World/receiver",
        name="receiver",
        waveLength = 1e-3,
        G = 10,
        e = 1,
        position=np.array([0,0,23]),
        scale=np.array([0.1,0.1,4]),
        height=1.0,
        color=np.array([1.0,0.0,0.0]),
    )
)

obstacle1 = world.scene.add(
    DynamicCuboid(
        prim_path="/World/obstacle1",
        name="obstacle1",
        position=np.array([5,5,25]),
        scale=np.array([19, 0.2, 48]),
        size=1.,
        color=np.array([0, 0, 1]),
    )
)

obstacle2 = world.scene.add(
    DynamicCuboid(
        prim_path="/World/obstacle2",
        name="obstacle2",
        position=np.array([-10, 0, 23]),
        scale=np.array([0.2, 23, 31]),
        size=1.,
        color=np.array([0, 0, 1]),
    )
)

world.render()
stage = get_current_stage()

# Run the application for multiple frames to ensure the synthetic data pipeline is initialized
timeline = get_timeline_interface()
timeline.play()
for _ in range(10):
    simulation_app.update()
timeline.pause()

# Get groundtruth
rgb_data = rgb.get_data()
depth_data = depth.get_data()
semantic_segmentation_data = semantic_segmentation.get_data()

In [ ]:
telecomPossible = np.zeros((81, 81))

for x in range(-40, 41):
    for y in range(-40, 41):
        # Move receiver to the new position
        receiver.set_world_pose(position=np.array([x, y, 23]))

        # Get target path from receiver
        target_path = receiver.prim_path

        # Compute the ray origin
        origin = emitter.moveOriginRaycast(receiver, 0.3)

        # Perform raycast using the Antenna method
        stage = get_current_stage()
        output_ray = emitter.performRaycast(receiver, stage, 0.3)

        # Check visibility conditions
        is_visible = (
            receiver.checkRaycastCoherence(output_ray, target_path) and
            receiver.checkPowerFeasibility(output_ray[1], 1e-6)
        )

        # Store the result in the map
        telecomPossible[x + 40, y + 40] = int(is_visible)


In [ ]:
plt.figure(dpi=300)
plt.imshow(telecomPossible, cmap='gray', extent=[-40,40,-40,40])
plt.scatter(0,0,color='green')
plt.title("Visibility of the receiver from the emitter point of view")
plt.xlabel("x (m)")
plt.ylabel("y (m)")
plt.savefig('Visibility.png')
plt.show()